In [2]:
import pandas as pd
import os
import numpy as np

In [3]:
df = pd.read_parquet("../5DATA/dataset/TRAIN_stage2")

# 다중공선성 check

In [4]:
X = df.drop(columns=["fraud"])
corr = X.corr().abs()

In [20]:
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={0: "corr"})
    .query("corr >= 0.8")
    .sort_values("corr", ascending=False)
)

high_corr_pairs

,level_0,level_1,corr
2127,cnt_past,client_weekday_prior_count,0.997876
18,id,tx_year,0.985827
4123,card_tx_1h,card_velocity_spike_ratio,0.980742
4013,client_tx_1h,velocity_spike_ratio,0.980253
3544,mcc_risk_level,vel_x_mcc_risk,0.974975
4736,amount_ratio_x_client_merchant_new,amount_ratio_x_client_mcc_new,0.970019
4728,client_merchant_is_new_x_mcc_smoothed_risk,client_mcc_is_new_x_mcc_smoothed_risk,0.961396
2983,per_capita_income,yearly_income,0.951558
4738,card_fraud_last3_x_client_merchant_new,client_fraud_last3_x_client_merchant_new,0.942714
4213,vel_x_error,has_error,0.933899


| Pair (corr)                                                                                 | Keep                                               | Drop                                       | Rationale                                                                                                   |
| ------------------------------------------------------------------------------------------- | -------------------------------------------------- | ------------------------------------------ | ----------------------------------------------------------------------------------------------------------- |
| cnt_past ↔ client_weekday_prior_count (0.9979)                                              | cnt_past                                           | client_weekday_prior_count                 | 둘 다 “과거 누적 카운트” 성격이 거의 동일. cnt_past가 더 범용적인 히스토리 길이 proxy라 유지.                                              |
| id ↔ tx_year (0.9858)                                                                       | tx_year                                            | id                                         | id는 식별자라 모델 입력에서 정보 누수/우연 상관 가능. 시간 정보는 tx_year로 유지.                                                        |
| card_tx_1h ↔ card_velocity_spike_ratio (0.9807)                                             | card_velocity_spike_ratio                          | card_tx_1h                                 | spike_ratio가 “현재 강도 / 기준”을 포함해 더 정보량이 큼.                                                                    |
| client_tx_1h ↔ velocity_spike_ratio (0.9803)                                                | velocity_spike_ratio                               | client_tx_1h                               | 위와 동일 논리. ratio 유지, raw count 제거.                                                                           |
| mcc_risk_level ↔ vel_x_mcc_risk (0.9750)                                                    | vel_x_mcc_risk                                     | mcc_risk_level                             | vel_x_mcc_risk는 mcc_risk_level을 포함(곱)하며 속도 컨텍스트까지 결합된 상호작용이라 유지.                                            |
| amount_ratio_x_client_merchant_new ↔ amount_ratio_x_client_mcc_new (0.9700)                 | amount_ratio_x_client_merchant_new                 | amount_ratio_x_client_mcc_new              | 둘 다 “amount_ratio × novelty” 계열. merchant novelty 쪽이 더 직접적인 이상거래 시그널인 경우가 많아 1개만 유지.                        |
| client_merchant_is_new_x_mcc_smoothed_risk ↔ client_mcc_is_new_x_mcc_smoothed_risk (0.9614) | client_merchant_is_new_x_mcc_smoothed_risk         | client_mcc_is_new_x_mcc_smoothed_risk      | 둘 다 mcc_risk 결합 novelty. merchant novelty 쪽만 남겨 중복 제거.                                                      |
| per_capita_income ↔ yearly_income (0.9516)                                                  | log_yearly_income (이미 있으면) / yearly_income         | per_capita_income                          | 둘 중 하나만. 소득은 스케일 안정성을 위해 log_yearly_income이 있으면 그걸 우선, 없으면 yearly_income 유지.                                |
| card_fraud_last3_x_client_merchant_new ↔ client_fraud_last3_x_client_merchant_new (0.9427)  | card_fraud_last3_x_client_merchant_new             | client_fraud_last3_x_client_merchant_new   | 둘 다 “히스토리 fraud × merchant new”. card 단위가 더 즉각적/거래 단위에 근접한 신호라 1개만 유지.                                      |
| vel_x_error ↔ has_error (0.9339)                                                            | vel_x_error                                        | has_error                                  | vel_x_error는 has_error를 포함하면서 속도 컨텍스트를 얹은 결합 피처. 중복 방지 위해 has_error 드랍(단, error 플래그가 필요한 해석/룰이면 keep).      |
| amount_vs_client_avg_ratio ↔ amount_ratio_x_client_merchant_new (0.9312)                    | amount_vs_client_avg_ratio                         | amount_ratio_x_client_merchant_new         | interaction이 ratio에 종속(또는 novelty와 강결합)이라 다중공선성 큼. 기본 ratio를 남기고 파생 interaction 제거.                         |
| amount_ratio_x_mcc_smoothed_risk ↔ amount_ratio_x_client_mcc_new (0.9201)                   | amount_ratio_x_mcc_smoothed_risk                   | amount_ratio_x_client_mcc_new              | 둘 다 amount_ratio 기반. risk 결합이 더 일반화되는 축이라 유지, novelty 결합은 제거.                                               |
| merchant_is_new_x_mcc_is_new ↔ vel_x_merchant_new (0.9158)                                  | vel_x_merchant_new                                 | merchant_is_new_x_mcc_is_new               | vel_x_merchant_new는 속도 컨텍스트 포함. 단순 곱(merchant_is_new×mcc_is_new)은 정보 중복이 큼.                                 |
| amount_vs_client_avg_ratio ↔ amount_ratio_x_client_mcc_new (0.9111)                         | amount_vs_client_avg_ratio                         | amount_ratio_x_client_mcc_new              | 위와 동일. 기본 ratio 유지.                                                                                         |
| amount_ratio_x_mcc_smoothed_risk ↔ amount_ratio_x_client_merchant_new (0.8977)              | amount_ratio_x_mcc_smoothed_risk                   | amount_ratio_x_client_merchant_new         | amount_ratio 파생 interaction 중 대표 1개만 유지. risk 결합만 남김.                                                       |
| current_age ↔ years_to_retirement (0.8976)                                                  | current_age                                        | years_to_retirement                        | 사실상 선형변환 관계. current_age가 더 직접적이며 누락/정의 변동이 적음.                                                             |
| card_fraud_last1 ↔ card_fraud_last3 (0.8897)                                                | card_fraud_last3                                   | card_fraud_last1                           | last3가 last1을 포함(누적)하므로 정보 우위. 1개만 유지.                                                                      |
| client_fraud_last1 ↔ client_fraud_last3 (0.8741)                                            | client_fraud_last3                                 | client_fraud_last1                         | 위와 동일.                                                                                                      |
| log_income_ratio_region ↔ income_ratio_region (0.8710)                                      | log_income_ratio_region                            | income_ratio_region                        | 로그 변환이 분포 안정/극단치 완화에 유리. 원본 비율 제거.                                                                          |
| mcc_smoothed_risk ↔ client_merchant_is_new_x_mcc_smoothed_risk (0.8627)                     | mcc_smoothed_risk                                  | client_merchant_is_new_x_mcc_smoothed_risk | interaction이 risk 신호를 강하게 재사용. risk(기본)를 남기고 interaction은 드랍(이미 amount_ratio_x_mcc_smoothed_risk 남긴 경우 특히). |
| client_merchant_is_new_x_log_interval_dev ↔ client_mcc_is_new_x_log_interval_dev (0.8552)   | client_merchant_is_new_x_log_interval_dev          | client_mcc_is_new_x_log_interval_dev       | 둘 다 interval_dev 결합 novelty. merchant novelty 쪽만 유지.                                                        |
| amount_vs_client_avg_ratio ↔ amount_ratio_x_mcc_smoothed_risk (0.8549)                      | amount_vs_client_avg_ratio                         | amount_ratio_x_mcc_smoothed_risk           | ratio와 강결합. amount_ratio 계열을 하나만 남긴다면 ratio를 우선 유지(혹은 반대로 risk 결합을 남기면 ratio를 드랍).                          |
| client_fraud_last1 ↔ card_fraud_last1 (0.8497)                                              | card_fraud_last1 (또는 card_fraud_last3 유지 시 둘 다 드랍) | client_fraud_last1                         | 단기(last1) 기준이라면 card 단위가 더 직접적. 이미 last3 유지 전략이면 last1 계열은 제거.                                              |
| log_yearly_income ↔ yearly_income (0.8350)                                                  | log_yearly_income                                  | yearly_income                              | 로그 변환 유지, 원본 제거.                                                                                            |
| mcc_smoothed_risk ↔ client_mcc_is_new_x_mcc_smoothed_risk (0.8298)                          | mcc_smoothed_risk                                  | client_mcc_is_new_x_mcc_smoothed_risk      | 위와 동일. risk(기본) 유지.                                                                                         |
| hour_sin ↔ tx_hour (0.8219)                                                                 | hour_sin + hour_cos                                | tx_hour                                    | cyclic 표현(사인/코사인)이 시간의 원형성을 보존. tx_hour 제거.                                                                 |
| cb_Visa ↔ cb_Mastercard (0.8184)                                                            | cb_Visa, cb_Mastercard (둘 다 유지)                    | (드랍 없음)                                    | 원-핫/더미는 “기준 카테고리”를 하나 빼야 상관이 줄어듦. 이미 다 포함했다면 기준 더미 1개만 드랍 권장(예: cb_Discover 또는 cb_Amex 중 하나).               |
| client_fraud_last3 ↔ card_fraud_last3 (0.8131)                                              | card_fraud_last3                                   | client_fraud_last3                         | 둘 다 장기 누적 신호. 카드 단위가 거래 단위에 더 가깝기 때문에 대표 1개만 남김(반대로 고객 중심이면 client 유지).                                     |




In [5]:
DROP_CANDIDATES = [
    # history count 중복
    "client_weekday_prior_count",

    # 식별자
    "id",

    # raw velocity count (ratio 유지 전략)
    "card_tx_1h",
    "client_tx_1h",

    # mcc risk raw (interaction 유지 전략)
    "mcc_risk_level",

    # amount interaction 중복
    "amount_ratio_x_client_mcc_new",
    "amount_ratio_x_client_merchant_new",

    # mcc interaction 중복
    "client_mcc_is_new_x_mcc_smoothed_risk",

    # income 중복
    "per_capita_income",
    "yearly_income",
    "income_ratio_region",

    # fraud interaction 중복
    "client_fraud_last3_x_client_merchant_new",

    # error raw (interaction 유지 시)
    "has_error",

    # merchant novelty 중복
    "merchant_is_new_x_mcc_is_new",

    # age 선형변환 중복
    "years_to_retirement",

    # fraud short history (last3 유지 전략)
    "card_fraud_last1",
    "client_fraud_last1",

    # log_interval interaction 중복
    "client_mcc_is_new_x_log_interval_dev",

    # cyclic 중복
    "tx_hour",

    # fraud long history 중 하나만 유지 (card 유지 전략)
    "client_fraud_last3",
]

In [6]:
df.drop(columns=[c for c in DROP_CANDIDATES if c in df.columns], inplace=True)

In [23]:
df.shape

(608430, 80)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 608430 entries, 0 to 608429
Data columns (total 80 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   refund_high_amount                          608430 non-null  int8   
 1   log_abs_amount                              608430 non-null  float32
 2   err_bad_cvv                                 608430 non-null  int8   
 3   card_hist_x_error                           608430 non-null  int8   
 4   client_hist_x_error                         608430 non-null  int8   
 5   err_bad_card_number                         608430 non-null  int8   
 6   err_insufficient_balance                    608430 non-null  int8   
 7   client_error_last5                          608430 non-null  int8   
 8   card_error_last5                            608430 non-null  int8   
 9   client_error_last3                          608430 non-null  int8   
 

# SHAP & Ablation

In [7]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

LABEL_COL = "fraud"  

y = df[LABEL_COL].astype(int)
X = df.drop(columns=[LABEL_COL])

# 1) 컬럼 타입 자동 추정

cat_cols = [c for c in X.columns if str(X[c].dtype) in ("object", "category")]
num_cols = [c for c in X.columns if c not in cat_cols]

print("num_cols:", len(num_cols), "cat_cols:", len(cat_cols))


# 2) Train/Valid split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) 간단 전처리 파이프

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)), 
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
    ],
    remainder="drop",
)


num_cols: 79 cat_cols: 0


In [26]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()

dtrain = lgb.Dataset(X_train_lgb, label=y_train, free_raw_data=False)
dvalid = lgb.Dataset(X_valid_lgb, label=y_valid, free_raw_data=False)

params = dict(
    objective="binary",
    metric=["auc", "average_precision"],
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=200,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    verbosity=-1,
)

bst = lgb.train(
    params,
    dtrain,
    num_boost_round=5000,
    valid_sets=[dvalid],
    valid_names=["valid"],
    callbacks=[
        lgb.early_stopping(200, verbose=True),
        lgb.log_evaluation(0),
    ],
)

print("best_iter:", bst.best_iteration)
print("best_score:", bst.best_score)

pred_valid = bst.predict(X_valid_lgb, num_iteration=bst.best_iteration)
print("LGB AUC:", roc_auc_score(y_valid, pred_valid))
print("LGB PR-AUC:", average_precision_score(y_valid, pred_valid))


# SHAP with tqdm (batch version)

from tqdm.auto import tqdm

sv = X_valid_lgb

batch_size = 2000
all_contrib = []

print("\nComputing SHAP (LightGBM native)...")

for i in tqdm(range(0, len(sv), batch_size)):
    batch = sv.iloc[i:i+batch_size]
    contrib = bst.predict(batch, pred_contrib=True)
    all_contrib.append(contrib[:, :-1])  # 마지막 열은 bias

shap_values = np.vstack(all_contrib)

imp = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=sv.columns
).sort_values(ascending=False)

print("\nTop SHAP features:\n", imp.head(30))


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[205]	valid's auc: 0.998993	valid's average_precision: 0.981192
best_iter: 205
best_score: defaultdict(<class 'collections.OrderedDict'>, {'valid': OrderedDict([('auc', np.float64(0.9989934804338919)), ('average_precision', np.float64(0.9811922405275116))])})
LGB AUC: 0.9989934804338918
LGB PR-AUC: 0.9811922405275115

Computing SHAP (LightGBM native)...


  0%|          | 0/61 [00:00<?, ?it/s]


Top SHAP features:
 mcc_smoothed_risk                             1.399850
hour_sin                                      0.215408
amount_ratio_x_mcc_smoothed_risk              0.173445
client_merchant_is_new_x_mcc_smoothed_risk    0.161154
hour_cos                                      0.153199
card_fraud_last3                              0.132100
log_abs_amount                                0.130290
card_merchant_is_new                          0.097048
tx_year                                       0.089624
card_velocity_spike_ratio                     0.071669
weekday                                       0.067152
log_interval_dev                              0.063950
seconds_since_prev_tx                         0.060550
client_cos_mean_past                          0.058091
is_highrisk_weekday                           0.053928
client_avg_interval_prev                      0.050716
amount_limit_ratio                            0.046476
amount_vs_client_avg_diff                   

| SHAP 방법                                    | 특징                      | 장점                                  | 단점                              | 적용 여부   | 선택 이유                                           |
| ------------------------------------------ | ----------------------- | ----------------------------------- | ------------------------------- | ------- | ----------------------------------------------- |
| KernelSHAP                                 | 모델-agnostic 근사 방식       | 어떤 모델에도 적용 가능                       | 계산량 큼, 근사 기반으로 분산 존재            | 사용하지 않음 | 트리 모델 검증 목적에 비해 과도하며 효율성 낮음                     |
| TreeExplainer (shap 패키지)                   | TreeSHAP 구현             | 정확한 Shapley value 계산                | 별도 explainer 객체 필요, 추가 계산 단계 존재 | 사용하지 않음 | LightGBM 내장 방식과 이론적으로 동일                        |
| LightGBM native SHAP (`pred_contrib=True`) | LightGBM 내부 TreeSHAP 구현 | 정확한 additive 분해, 트리 구조 직접 반영, 구현 간결 | LightGBM 전용                     | 사용      | 트리 기반 비선형 모델에서 EDA로 설계한 피처의 실제 기여도를 직접 검증하기에 적합 |

| 선택 기준             | 설명                                           |
| ----------------- | -------------------------------------------- |
| 기여도 정량화           | 각 feature가 예측값(log-odds)에 얼마나 기여하는지 직접 계산 가능 |
| 모델 구조 반영          | 트리 기반 비선형 패턴을 그대로 반영                         |
| Interaction 검증    | EDA에서 설계한 상호작용 피처의 실제 영향력 확인 가능              |
| 통계 분석과 비교         | 로지스틱 OR 분석 결과와 교차 검증 가능                      |
| Feature 검증 목적 적합성 | 모델 설명이 아니라 피처의 구조적 유효성 검증을 위한 분석 도구로 적합      |


# Attention

In [27]:
def check_finite(name, arr):
    arr = np.asarray(arr)
    n_nan = np.isnan(arr).sum()
    n_inf = np.isinf(arr).sum()
    print(f"{name}: nan={n_nan}, inf={n_inf}, shape={arr.shape}, dtype={arr.dtype}")

check_finite("X_train raw", X_train.values)
check_finite("X_valid raw", X_valid.values)
check_finite("y_train", y_train.values)
check_finite("y_valid", y_valid.values)

X_train raw: nan=0, inf=0, shape=(486744, 79), dtype=float32
X_valid raw: nan=0, inf=0, shape=(121686, 79), dtype=float32
y_train: nan=0, inf=0, shape=(486744,), dtype=int64
y_valid: nan=0, inf=0, shape=(121686,), dtype=int64


In [28]:
nan_col_tr = X_train.isna().sum().sort_values(ascending=False)
nan_col_va = X_valid.isna().sum().sort_values(ascending=False)

print("=== Train NaN cols (top) ===")
print(nan_col_tr[nan_col_tr > 0].head(30))
print("\n#cols with NaN (train):", int((nan_col_tr > 0).sum()))
print("\n=== Valid NaN cols (top) ===")
print(nan_col_va[nan_col_va > 0].head(30))
print("\n#cols with NaN (valid):", int((nan_col_va > 0).sum()))

=== Train NaN cols (top) ===
Series([], dtype: int64)

#cols with NaN (train): 0

=== Valid NaN cols (top) ===
Series([], dtype: int64)

#cols with NaN (valid): 0


In [8]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

feature_names = list(X_train.columns)
n_features = len(feature_names)

scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train.values.astype(np.float32))
Xva = scaler.transform(X_valid.values.astype(np.float32))

ytr = y_train.values.astype(np.float32)
yva = y_valid.values.astype(np.float32)

class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_loader = DataLoader(TabDataset(Xtr, ytr), batch_size=4096, shuffle=True, num_workers=0)
valid_loader = DataLoader(TabDataset(Xva, yva), batch_size=8192, shuffle=False, num_workers=0)

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1) Attention layer (weights 반환)

class AttnEncoderLayer(nn.Module):
    def __init__(self, d_model=64, n_heads=4, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, return_attn=False):
        # x: [B, T, D]
        attn_out, attn_w = self.mha(x, x, x, need_weights=True, average_attn_weights=False)
        x = self.ln1(x + attn_out)
        x = self.ln2(x + self.ff(x))
        if return_attn:
            # attn_w: [B, heads, T, T]
            return x, attn_w
        return x


# 2) Tabular Transformer (CLS 토큰 사용)

class TabularAttentionModel(nn.Module):
    def __init__(self, n_features, d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model

        # feature별 1->d 투영 (각 feature마다 별도의 linear)
        self.feat_proj = nn.ModuleList([nn.Linear(1, d_model) for _ in range(n_features)])

        # CLS token
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

        self.layers = nn.ModuleList([AttnEncoderLayer(d_model, n_heads, dropout) for _ in range(n_layers)])

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, X, return_attn=False):
        # X: [B, F]
        B, F = X.shape

        # feature tokens 만들기: [B, F, D]
        toks = []
        for j in range(F):
            xj = X[:, j:j+1]                 # [B, 1]
            toks.append(self.feat_proj[j](xj))  # [B, D]
        tok = torch.stack(toks, dim=1)       # [B, F, D]

        # CLS 붙이기: [B, 1+F, D]
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, tok], dim=1)

        attn_all = []
        for layer in self.layers:
            if return_attn:
                x, attn = layer(x, return_attn=True)
                attn_all.append(attn)
            else:
                x = layer(x, return_attn=False)

        # CLS representation으로 예측
        cls_repr = x[:, 0, :]               # [B, D]
        logit = self.head(cls_repr).squeeze(1)

        if return_attn:
            return logit, attn_all  # list of [B, heads, T, T]
        return logit


# 3) 학습 루프

model = TabularAttentionModel(n_features=n_features, d_model=64, n_heads=4, n_layers=2, dropout=0.1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

def eval_model():
    model.eval()
    ps, ys = [], []
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            logit = model(xb)
            prob = torch.sigmoid(logit).cpu().numpy()
            ps.append(prob)
            ys.append(yb.numpy())
    p = np.concatenate(ps)
    t = np.concatenate(ys)
    return roc_auc_score(t, p), average_precision_score(t, p)

EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    model.train()
    pbar = tqdm(train_loader, desc=f"train epoch {epoch}", leave=False)
    for xb, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        logit = model(xb)
        loss = loss_fn(logit, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        pbar.set_postfix(loss=float(loss.detach().cpu()))

    auc, pr = eval_model()
    print(f"[epoch {epoch}] valid AUC={auc:.5f}  PR-AUC={pr:.5f}")


# 4) Attention 추출 → 컬럼 중요도
# - CLS(0번 토큰)에서 각 feature 토큰으로 가는 attention을 사용

def extract_feature_attention_importance(model, loader, n_batches=50):
    model.eval()
    # 누적: feature별 attention 합
    att_sum = np.zeros((n_features,), dtype=np.float64)
    cnt = 0

    with torch.no_grad():
        for b, (xb, yb) in enumerate(tqdm(loader, desc="extract attention", total=min(n_batches, len(loader)))):
            if b >= n_batches:
                break
            xb = xb.to(device)

            logit, attn_all = model(xb, return_attn=True)
            attn = attn_all[-1]  # [B, heads, T, T]

            # CLS -> feature 토큰 attention: query=0, key=1..F
            # shape: [B, heads, F]
            cls_to_feat = attn[:, :, 0, 1:]  

            # heads 평균, batch 평균 → [F]
            score = cls_to_feat.mean(dim=1).mean(dim=0).cpu().numpy()
            att_sum += score
            cnt += 1

    att_mean = att_sum / max(cnt, 1)
    imp = pd.Series(att_mean, index=feature_names).sort_values(ascending=False)
    return imp

att_imp = extract_feature_attention_importance(model, valid_loader, n_batches=50)
print("\nTop Attention features:\n", att_imp.head(30))


train epoch 1:   0%|          | 0/119 [00:00<?, ?it/s]

[epoch 1] valid AUC=0.99296  PR-AUC=0.92121


train epoch 2:   0%|          | 0/119 [00:00<?, ?it/s]

[epoch 2] valid AUC=0.99524  PR-AUC=0.93733


train epoch 3:   0%|          | 0/119 [00:00<?, ?it/s]

...

[epoch 4] valid AUC=0.99624  PR-AUC=0.95301


train epoch 5:   0%|          | 0/119 [00:00<?, ?it/s]

[epoch 5] valid AUC=0.99639  PR-AUC=0.95504


extract attention:   0%|          | 0/15 [00:00<?, ?it/s]


Top Attention features:
 card_fraud_last3                              0.131501
high_vel_error                                0.030661
client_error_last3                            0.025126
tx_day                                        0.019401
hour_sin                                      0.018729
vel_x_client_mcc_new                          0.017744
is_prepaid                                    0.017519
client_weekday_match_last1                    0.017001
high_mcc_high_weekday                         0.016798
client_merchant_is_new_x_mcc_smoothed_risk    0.016525
log_interval_dev                              0.015333
log_income_ratio_region                       0.015064
cnt_past                                      0.014896
merchant_change_cnt_last5                     0.014825
amt_over_q99                                  0.014715
high_vel_flag                                 0.014658
vel_x_card_mcc_new                            0.014625
err_bad_cvv                            

In [31]:
compare = pd.DataFrame({
    "shap_mean_abs": imp,        
    "attn_cls2feat": att_imp
}).fillna(0.0)

compare["shap_rank"] = compare["shap_mean_abs"].rank(ascending=False, method="min")
compare["attn_rank"] = compare["attn_cls2feat"].rank(ascending=False, method="min")
compare["rank_gap"] = compare["attn_rank"] - compare["shap_rank"]

print(compare.sort_values("shap_mean_abs", ascending=False).head(40))


                                            shap_mean_abs  attn_cls2feat  \
mcc_smoothed_risk                                1.399850       0.006880   
hour_sin                                         0.215408       0.012307   
amount_ratio_x_mcc_smoothed_risk                 0.173445       0.002820   
client_merchant_is_new_x_mcc_smoothed_risk       0.161154       0.020162   
hour_cos                                         0.153199       0.004406   
card_fraud_last3                                 0.132100       0.003327   
log_abs_amount                                   0.130290       0.010111   
card_merchant_is_new                             0.097048       0.005757   
tx_year                                          0.089624       0.014161   
card_velocity_spike_ratio                        0.071669       0.017046   
weekday                                          0.067152       0.007780   
log_interval_dev                                 0.063950       0.010761   
seconds_sinc

## Keep 후보 컬럼

| Feature                                    | 근거                       |
| ------------------------------------------ | ------------------------ |
| mcc_smoothed_risk                          | SHAP 1위, 모델 핵심 리스크 축     |
| hour_sin                                   | 시간 패턴 상위 중요도             |
| hour_cos                                   | 시간 패턴 보조 축               |
| client_merchant_is_new_x_mcc_smoothed_risk | SHAP·Attention 모두 상위     |
| amount_ratio_x_mcc_smoothed_risk           | SHAP 상위 interaction 핵심   |
| card_fraud_last3                           | Fraud history 핵심 변수      |
| card_fraud_last3_x_client_merchant_new     | 상호작용 중요도 양호              |
| log_abs_amount                             | 기본 거래 강도 변수              |
| card_velocity_spike_ratio                  | SHAP·Attention 모두 상위     |
| velocity_spike_ratio                       | Attention 상위             |
| log_interval_dev                           | 거래 간격 이상 탐지              |
| seconds_since_prev_tx                      | 시간 간격 정보                 |
| merchant_change_cnt_last5                  | Attention 상위, 행동 변화 반영   |
| vel_x_mcc_risk                             | 속도 × MCC 리스크 interaction |
| amount_limit_ratio                         | 한도 대비 금액 정보              |
| amount_vs_client_avg_diff                  | 개인 평균 대비 deviation       |
| client_cos_mean_past                       | 고객 행동 패턴 요약              |
| client_sin_mean_past                       | 고객 행동 패턴 요약              |
| tx_year                                    | 시간 drift 신호 가능           |
| credit_score                               | Attention 상위, 개인 신용 맥락   |

---

## Drop 후보 컬럼

| Feature                       | 이유                                |
| ----------------------------- | --------------------------------- |
| client_avg_interval_prev      | velocity 및 interval_dev와 정보 중복 가능 |
| num_credit_cards              | 기여도 낮음                            |
| tx_month                      | 시간 정보 중복 가능                       |
| tx_day                        | 시간 정보 중복 가능                       |
| log_income_ratio_region       | 기여도 낮음                            |
| current_age                   | 상대적 중요도 낮음                        |
| total_debt                    | 기여도 낮음                            |
| amount_income_ratio           | SHAP·Attention 모두 낮음              |
| amount_vs_client_quantile_q95 | 기여도 낮음                            |
| amount_vs_client_quantile_q99 | 기여도 낮음                            |
| credit_limit                  | amount_limit_ratio와 중복 가능         |
| log_yearly_income             | income_ratio_region과 중복 가능        |
| cnt_past                      | 간접적 지표, 직접적 설명력 낮음                |

---

## 추가 검증

| Feature                   | 코멘트                           |
| ------------------------- | ----------------------------- |
| sin_shift                 | Attention 상위, 모델 구조 의존 가능성 있음 |
| months_from_account       | Attention 상대적으로 높음            |
| total_debt                | 일부 모델에서 보조 역할 가능              |
| velocity_spike_ratio      | 파생 변수와 중복 여부 검증 필요            |
| tx_year                   | Drift 반영 변수로 유지 여부 실험 권장      |
| merchant_change_cnt_last5 | 행동 변화 변수로 유지 가능성 높음           |


---

## Ablation Test

In [11]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

params = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=200,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    verbosity=-1,
)

def train_eval(X_tr, y_tr, X_va, y_va):
    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    dvalid = lgb.Dataset(X_va, label=y_va, free_raw_data=False)

    bst = lgb.train(
        params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    pred = bst.predict(X_va, num_iteration=bst.best_iteration)

    return {
        "auc": roc_auc_score(y_va, pred),
        "prauc": average_precision_score(y_va, pred),
        "best_iter": bst.best_iteration,
    }


In [12]:
base_result = train_eval(X_train, y_train, X_valid, y_valid)
print("BASE:", base_result)

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[205]	valid_0's auc: 0.998993
BASE: {'auc': 0.9989934804338918, 'prauc': 0.9811922405275115, 'best_iter': 205}


In [13]:
drop_one_results = []

for col in tqdm(X_train.columns):
    cols = [c for c in X_train.columns if c != col]

    res = train_eval(
        X_train[cols],
        y_train,
        X_valid[cols],
        y_valid,
    )

    drop_one_results.append({
        "dropped_feature": col,
        "auc_drop": base_result["auc"] - res["auc"],
        "prauc_drop": base_result["prauc"] - res["prauc"],
        "auc": res["auc"],
        "prauc": res["prauc"],
    })

drop_one_df = pd.DataFrame(drop_one_results)\
    .sort_values("auc_drop", ascending=False)

drop_one_df

  0%|          | 0/79 [00:00<?, ?it/s]

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[507]	valid_0's auc: 0.999116
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[471]	valid_0's auc: 0.999026
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[688]	valid_0's auc: 0.999114
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[519]	valid_0's auc: 0.999046
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[390]	valid_0's auc: 0.999012
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[372]	valid_0's auc: 0.99906
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[606]	valid_0's auc: 0.999072
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[282]	valid_0's

,dropped_feature,auc_drop,prauc_drop,auc,prauc
70,card_fraud_last3,0.000195,0.006691,0.998798,0.974501
16,tx_month,0.000105,-0.000746,0.998888,0.981939
17,tx_year,0.000091,0.001333,0.998903,0.979859
58,vel_x_client_mcc_new,0.000080,-0.001178,0.998914,0.982370
31,current_age,0.000077,-0.000152,0.998916,0.981344
...,...,...,...,...,...
25,client_sin_mean_past,-0.000155,-0.001628,0.999149,0.982820
9,client_error_last3,-0.000180,-0.002422,0.999174,0.983614
77,card_fraud_last3_x_client_merchant_new,-0.000189,-0.002242,0.999183,0.983434
46,card_merchant_is_new,-0.000193,-0.002275,0.999186,0.983467


In [14]:
drop_one_df.to_csv("drop_one_results.csv", index=False)

### Drop-One Ablation Summary (AUC 기준)

**기준**

* `auc_drop > 0` : 해당 피처 제거 시 성능 하락 → **KEEP 권장**
* `auc_drop ≤ 0` : 제거해도 성능 유지/개선 → **DROP 후보**

---

## 1. KEEP (auc_drop > 0)

| Feature                   | AUC Drop | PR-AUC Drop | 판단    |
| ------------------------- | -------- | ----------- | ----- |
| card_fraud_last3          | 0.000195 | 0.006691    | 핵심 피처 |
| tx_month                  | 0.000105 | -0.000746   | 유지    |
| tx_year                   | 0.000091 | 0.001333    | 유지    |
| vel_x_client_mcc_new      | 0.000080 | -0.001178   | 유지    |
| current_age               | 0.000077 | -0.000152   | 유지    |
| amt_over_q95              | 0.000075 | 0.000415    | 유지    |
| client_error_last5        | 0.000047 | -0.000813   | 유지    |
| amount_vs_client_avg_diff | 0.000043 | -0.000649   | 유지    |
| total_debt                | 0.000040 | -0.001437   | 유지    |
| vel_x_high_mcc            | 0.000039 | -0.000046   | 유지    |
| cos_shift                 | 0.000037 | -0.000240   | 유지    |
| months_from_account       | 0.000025 | -0.000930   | 유지    |
| is_prepaid                | 0.000017 | 0.001266    | 유지    |
| mcc_smoothed_risk         | 0.000004 | 0.001151    | 유지    |

---

## 2. 영향 미미 (auc_drop ≈ 0)

| Feature                       |
| ----------------------------- |
| discover_x_cvv                |
| vel_x_merchant_new            |
| merchant_is_new_x_has_error   |
| prepaid_logamount_interaction |

→ 중복 또는 다른 interaction에 흡수되었을 가능성 있음.

---

## 3. DROP 후보 (auc_drop < 0)

| Feature                                    | AUC Drop  | PR-AUC Drop |
| ------------------------------------------ | --------- | ----------- |
| amount_limit_ratio                         | -0.000002 | -0.000833   |
| hour_sin                                   | -0.000004 | 0.000722    |
| vel_x_card_mcc_new                         | -0.000008 | -0.001234   |
| credit_score                               | -0.000014 | -0.001568   |
| amount_vs_client_quantile_q99              | -0.000016 | 0.000180    |
| vel_x_error                                | -0.000017 | -0.001412   |
| client_hist_x_error                        | -0.000019 | -0.001557   |
| cb_Discover                                | -0.000021 | -0.001108   |
| client_cos_mean_past                       | -0.000027 | -0.001037   |
| card_velocity_spike_ratio                  | -0.000028 | -0.000869   |
| high_vel_flag                              | -0.000029 | -0.000829   |
| log_abs_amount                             | -0.000032 | -0.001639   |
| credit_limit                               | -0.000034 | 0.000439    |
| client_weekday_match_last1                 | -0.000038 | -0.002594   |
| cb_Visa                                    | -0.000038 | -0.000537   |
| cb_Amex                                    | -0.000041 | -0.002314   |
| amount_income_ratio                        | -0.000042 | -0.001372   |
| amount_vs_client_quantile_q95              | -0.000044 | -0.001718   |
| seconds_since_prev_tx                      | -0.000045 | -0.000175   |
| amt_over_q99                               | -0.000045 | -0.001815   |
| card_hist_x_error                          | -0.000053 | -0.001652   |
| has_chip                                   | -0.000055 | -0.001469   |
| hour_cos                                   | -0.000057 | -0.000724   |
| card_error_last1                           | -0.000059 | -0.000897   |
| cb_Mastercard                              | -0.000060 | -0.001724   |
| err_bad_card_number                        | -0.000066 | -0.001077   |
| amount_vs_recent_window_avg                | -0.000067 | -0.001739   |
| high_mcc_high_weekday                      | -0.000068 | -0.001818   |
| cnt_past                                   | -0.000068 | -0.000005   |
| weekday                                    | -0.000069 | -0.000443   |
| client_merchant_is_new_x_log_interval_dev  | -0.000069 | -0.000463   |
| merchant_change_cnt_last5                  | -0.000074 | -0.002829   |
| client_tx_1h_avg_prev                      | -0.000077 | -0.001026   |
| err_insufficient_balance                   | -0.000079 | -0.001945   |
| sin_shift                                  | -0.000084 | -0.001848   |
| log_interval_dev                           | -0.000090 | -0.002280   |
| is_credit                                  | -0.000094 | -0.001855   |
| high_vel_new_mcc                           | -0.000095 | -0.002264   |
| log_income_ratio_region                    | -0.000095 | -0.001959   |
| velocity_spike_ratio                       | -0.000096 | -0.001360   |
| client_merchant_is_new_x_mcc_smoothed_risk | -0.000097 | -0.000973   |
| client_avg_interval_prev                   | -0.000103 | -0.001892   |
| amount_ratio_x_mcc_smoothed_risk           | -0.000103 | -0.001623   |
| high_vel_error                             | -0.000104 | -0.001880   |
| client_merchant_is_new                     | -0.000107 | -0.001640   |
| vel_x_mcc_risk                             | -0.000112 | -0.002045   |
| card_error_last3                           | -0.000115 | -0.001652   |
| log_yearly_income                          | -0.000116 | -0.001751   |
| client_error_last1                         | -0.000118 | -0.002019   |
| err_bad_cvv                                | -0.000120 | -0.002240   |
| refund_high_amount                         | -0.000123 | -0.002376   |
| client_weekday_prev                        | -0.000128 | -0.002870   |
| is_highrisk_weekday                        | -0.000138 | -0.001364   |
| num_credit_cards                           | -0.000143 | -0.002735   |
| card_error_last5                           | -0.000152 | -0.002654   |
| client_sin_mean_past                       | -0.000155 | -0.001628   |
| client_error_last3                         | -0.000180 | -0.002422   |
| card_fraud_last3_x_client_merchant_new     | -0.000189 | -0.002242   |
| card_merchant_is_new                       | -0.000193 | -0.002275   |
| tx_day                                     | -0.000211 | -0.001698   |

---

## 최종 Drop 컬럼 (Keep 후보 리스트 ∩ Drop-One Ablation에서 auc_drop ≤ 0)

| Feature                                    | 근거 요약                                               |
| ------------------------------------------ | --------------------------------------------------- |
| hour_sin                                   | Drop-one에서 제거해도 AUC 개선(auc_drop < 0), PR-AUC는 소폭 증가 |
| hour_cos                                   | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| client_merchant_is_new_x_mcc_smoothed_risk | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| amount_ratio_x_mcc_smoothed_risk           | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| card_fraud_last3_x_client_merchant_new     | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| log_abs_amount                             | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| card_velocity_spike_ratio                  | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| velocity_spike_ratio                       | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| log_interval_dev                           | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| seconds_since_prev_tx                      | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| merchant_change_cnt_last5                  | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| vel_x_mcc_risk                             | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| amount_limit_ratio                         | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| client_cos_mean_past                       | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| client_sin_mean_past                       | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |
| credit_score                               | Drop-one에서 제거해도 AUC 개선(auc_drop < 0)                |

---

In [16]:
FINAL_DROP_COLUMNS = [
    # =========================
    # 1. 기존 Drop Candidates
    # =========================

    # history count 중복
    "client_weekday_prior_count",

    # 식별자
    "id",

    # raw velocity count (ratio 유지 전략)
    "card_tx_1h",
    "client_tx_1h",

    # mcc risk raw (interaction 유지 전략)
    "mcc_risk_level",

    # amount interaction 중복
    "amount_ratio_x_client_mcc_new",
    "amount_ratio_x_client_merchant_new",

    # mcc interaction 중복
    "client_mcc_is_new_x_mcc_smoothed_risk",

    # income 중복
    "per_capita_income",
    "yearly_income",
    "income_ratio_region",

    # fraud interaction 중복
    "client_fraud_last3_x_client_merchant_new",

    # error raw (interaction 유지 시)
    "has_error",

    # merchant novelty 중복
    "merchant_is_new_x_mcc_is_new",

    # age 선형변환 중복
    "years_to_retirement",

    # fraud short history (last3 유지 전략)
    "card_fraud_last1",
    "client_fraud_last1",

    # log_interval interaction 중복
    "client_mcc_is_new_x_log_interval_dev",

    # cyclic 중복
    "tx_hour",

    # fraud long history 중 하나만 유지 (card 유지 전략)
    "client_fraud_last3",


    # =========================
    # 2. Drop-One 기반 추가 Drop
    # =========================

    "hour_sin",
    "hour_cos",
    "client_merchant_is_new_x_mcc_smoothed_risk",
    "amount_ratio_x_mcc_smoothed_risk",
    "card_fraud_last3_x_client_merchant_new",
    "log_abs_amount",
    "card_velocity_spike_ratio",
    "velocity_spike_ratio",
    "log_interval_dev",
    "seconds_since_prev_tx",
    "merchant_change_cnt_last5",
    "vel_x_mcc_risk",
    "amount_limit_ratio",
    "client_cos_mean_past",
    "client_sin_mean_past",
    "credit_score",
]